In [1]:
# Imprimir información sobre la división
import pickle
with open("split_data_dict.pkl", "rb") as f:
    obj = pickle.load(f)

prueba = obj["prueba"]
split_data_dict = obj["split"]

for key, data in split_data_dict.items():
    print(f"Dataset: {key}")
    print(f"Tamaño de entrenamiento: {1 - prueba} | prueba: {prueba}")
    print(f"Min/Max de X_train: {data['train']['X'].min()}, {data['train']['X'].max()}")
    print(f"Min/Max de X_test: {data['test']['X'].min()}, {data['test']['X'].max()}")
    print(f"Tamaño -> X_train: {data['train']['X'].shape}, X_test: {data['test']['X'].shape}, "
          f"Y_train: {data['train']['Y'].shape}, Y_test: {data['test']['Y'].shape}\n")

Dataset: orig
Tamaño de entrenamiento: 0.8 | prueba: 0.2
Min/Max de X_train: 0.9745220898716793, 3.9621314639076735
Min/Max de X_test: 0.9748580503748066, 3.726623659510136
Tamaño -> X_train: (5756, 501), X_test: (1440, 501), Y_train: (5756, 1), Y_test: (1440, 1)

Dataset: sg
Tamaño de entrenamiento: 0.8 | prueba: 0.2
Min/Max de X_train: 0.9752886127765829, 3.5910473108689374
Min/Max de X_test: 0.9755120416313424, 3.4683186516323525
Tamaño -> X_train: (5756, 501), X_test: (1440, 501), Y_train: (5756, 1), Y_test: (1440, 1)

Dataset: msc
Tamaño de entrenamiento: 0.8 | prueba: 0.2
Min/Max de X_train: 1.0404674721382379, 3.6052977194138074
Min/Max de X_test: 1.0405483671078637, 3.396062387918411
Tamaño -> X_train: (5756, 501), X_test: (1440, 501), Y_train: (5756, 1), Y_test: (1440, 1)

Dataset: snv
Tamaño de entrenamiento: 0.8 | prueba: 0.2
Min/Max de X_train: -2.811475760461824, 2.2578953916480784
Min/Max de X_test: -2.8084884803188337, 1.846030072642991
Tamaño -> X_train: (5756, 501), X_

## Encontrando los mejores hiperparámetros
### SVR 

In [2]:
import torch
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Usando GPU:", torch.cuda.get_device_name(0))


CUDA disponible: True
Usando GPU: NVIDIA GeForce RTX 4090 Laptop GPU


In [5]:
import optuna
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import time
from tqdm import tqdm

optuna.logging.set_verbosity(optuna.logging.WARNING)

itera = 200
seed = 15
kfold = KFold(n_splits=5, random_state=seed, shuffle=True)

best_hyperparams = {"svr": {}}


def optimize_hyperparameters_svr(X_train, y_train):

    def objective(trial):
        start = time.time()

        C = trial.suggest_float('C', 1, 30.0, log=True)
        epsilon = trial.suggest_float('epsilon', 0.01, 1.0, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'sigmoid'])

        gamma = None
        if kernel in ['rbf', 'sigmoid']:
            gamma = trial.suggest_float('gamma', 1e-4, 10, log=True)

        model = Pipeline([
            ('scaler', StandardScaler()),
            ('svr', SVR(C=C, epsilon=epsilon, kernel=kernel,
                        gamma=gamma if gamma else 'scale'))
        ])

        scores = cross_val_score(
            model, X_train, y_train,
            cv=kfold,
            scoring='neg_mean_squared_error',
            n_jobs=-1
        )

        mse = -scores.mean()

        print(f"Trial {trial.number} | MSE: {mse:.5f} | Tiempo: {time.time() - start:.2f}s")

        return mse

    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=2025),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=20,
            n_warmup_steps=5
        )
    )

    print("Optimizando...")

    study.optimize(
        objective,
        n_trials=itera,
        n_jobs=1,  
        show_progress_bar=True
    )

    return study.best_params, study.best_value

C:\Users\mjime\anaconda3\envs\tesiskarlo\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



CPU Núcleos: [0.4, 0.0, 0.1, 0.0, 0.2, 0.0, 0.0, 0.0, 2.8, 0.0, 6.0, 0.1, 0.0, 0.0, 1.2, 0.1, 0.2, 0.0, 0.8, 0.5, 0.5, 0.0, 0.1, 0.0, 0.1, 0.0, 0.2, 0.0, 0.2, 0.3, 0.0, 0.0]
RAM Uso: 33.4%
GPU Uso: 0.0% | Mem: 4.3% | Temp: 46.0°C

CPU Núcleos: [0.1, 0.0, 0.1, 0.0, 0.1, 0.0, 0.0, 0.0, 2.3, 0.1, 0.7, 2.2, 0.0, 0.0, 0.1, 0.2, 0.0, 0.0, 0.2, 0.6, 0.2, 0.2, 0.1, 0.1, 0.1, 0.1, 0.1, 0.0, 0.1, 0.2, 0.3, 0.1]
RAM Uso: 33.4%
GPU Uso: 0.0% | Mem: 4.4% | Temp: 45.0°C

CPU Núcleos: [0.2, 0.0, 0.4, 0.0, 0.1, 0.0, 0.2, 0.0, 2.7, 0.0, 0.8, 1.3, 0.0, 0.0, 0.1, 0.2, 0.2, 0.0, 0.0, 0.4, 0.4, 0.2, 0.0, 0.0, 0.2, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
RAM Uso: 33.3%
GPU Uso: 0.0% | Mem: 4.4% | Temp: 44.0°C

CPU Núcleos: [0.5, 0.1, 0.0, 0.0, 0.1, 0.0, 0.1, 0.0, 3.7, 0.0, 0.9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.7, 0.3, 0.0, 0.0, 0.3, 0.0, 0.0, 0.0]
RAM Uso: 33.3%
GPU Uso: 0.0% | Mem: 4.3% | Temp: 44.0°C

CPU Núcleos: [0.0, 0.4, 0.2, 0.1, 0.0, 0.1, 0.0, 0.1, 0.3, 2.3, 1.7, 0.0, 0

In [6]:
# Ejecutar optimización
for dataset in split_data_dict.keys():
    X_train = split_data_dict[dataset]['train']['X']
    y_train = split_data_dict[dataset]['train']['Y'].ravel()

    print(f"\n==============================")
    print(f"Optimizando SVR para {dataset}")
    print("==============================\n")

    best_params, best_value = optimize_hyperparameters_svr(X_train, y_train)

    best_hyperparams["svr"][dataset] = {
        "best_params": best_params,
        "best_value": best_value
    }

# Detener monitor
monitor_flag = False
monitor_thread.join()

# Resultados
print("\n=== Mejores Hiperparámetros ===")
for dataset, result in best_hyperparams["svr"].items():
    print(f"\nDataset: {dataset}")
    print("Hiperparámetros:", result["best_params"])
    print(f"Mejor MSE: {result['best_value']:.5f}")



Optimizando SVR para orig

Optimizando...


  0%|                                                                                          | 0/200 [00:00<?, ?it/s]


CPU Núcleos: [6.6, 1.4, 3.7, 1.5, 21.8, 1.5, 14.5, 1.9, 18.2, 1.9, 19.8, 2.0, 14.6, 1.9, 14.0, 1.6, 3.0, 2.9, 3.7, 3.3, 3.1, 3.2, 2.8, 3.0, 4.0, 3.3, 3.0, 2.9, 3.7, 3.0, 3.0, 2.8]
RAM Uso: 41.4%
GPU Uso: 0.0% | Mem: 4.5% | Temp: 48.0°C


Best trial: 0. Best value: 68.7057:   0%|▏                                           | 1/200 [00:20<1:06:22, 20.01s/it]

Trial 0 | MSE: 68.70567 | Tiempo: 20.01s

CPU Núcleos: [4.0, 0.1, 0.6, 9.8, 34.2, 0.1, 10.2, 0.4, 11.0, 0.0, 1.5, 15.2, 14.2, 0.0, 11.4, 0.0, 2.5, 0.9, 0.2, 21.1, 13.7, 11.1, 3.0, 0.6, 9.0, 6.7, 3.3, 0.9, 15.8, 10.2, 2.3, 0.8]
RAM Uso: 45.0%
GPU Uso: 0.0% | Mem: 4.5% | Temp: 53.0°C


...

Best trial: 163. Best value: 20.7175: 100%|███████████████████████████████████████▊| 199/200 [1:02:22<00:10, 10.00s/it]

Trial 198 | MSE: 43.65959 | Tiempo: 7.95s


Best trial: 163. Best value: 20.7175: 100%|████████████████████████████████████████| 200/200 [1:02:28<00:00, 18.74s/it]

Trial 199 | MSE: 13845316.56111 | Tiempo: 6.64s



=== Mejores Hiperparámetros ===

Dataset: orig
Hiperparámetros: {'C': 29.958171631189153, 'epsilon': 0.04455338175146469, 'kernel': 'rbf', 'gamma': 0.007460250384322396}
Mejor MSE: 21.35367

Dataset: sg
Hiperparámetros: {'C': 28.707297092891597, 'epsilon': 0.4504506686358284, 'kernel': 'rbf', 'gamma': 0.01226412943539597}
Mejor MSE: 15.80649

Dataset: msc
Hiperparámetros: {'C': 29.688935279244866, 'epsilon': 0.06659234585007497, 'kernel': 'rbf', 'gamma': 0.0024710740103106917}
Mejor MSE: 21.15390

Dataset: snv
Hiperparámetros: {'C': 23.6778133149059, 'epsilon': 0.036976553722841324, 'kernel': 'rbf', 'gamma': 0.0028254914500518034}
Mejor MSE: 20.71747


### Random Forest

In [5]:
import os, time, pickle
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold

optuna.logging.set_verbosity(optuna.logging.WARNING)

itera = 200
seed = 15
cpu_count = os.cpu_count() or 1
per_trial_threads = max(1, cpu_count - 2)   # dejar 2 hilos para el sistema
kfold = KFold(n_splits=5, random_state=seed, shuffle=True)

best_hyperparams = {"random_forest": {}}

def optimize_hyperparameters_rf(X_train, y_train):
    def objective(trial):
        start = time.time()
        n_estimators = trial.suggest_int('n_estimators', 50, 500, step=10)
        max_depth = trial.suggest_int('max_depth', 5, 50, step=5)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 5)
        max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])

        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=seed,
            n_jobs=per_trial_threads
        )

        # evitar nested parallelism: estimator usa threads, así que cross_val_score n_jobs=1
        scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring='neg_mean_squared_error', n_jobs=1)
        mse = -scores.mean()
        print(f"Trial {trial.number} | n_est={n_estimators} max_depth={max_depth} | MSE: {mse:.5f} | Tiempo: {time.time()-start:.1f}s")
        return mse

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=2025),
                               pruner=optuna.pruners.MedianPruner(n_startup_trials=20, n_warmup_steps=5))
    study.optimize(objective, n_trials=itera, n_jobs=1, show_progress_bar=True)
    return study.best_params, study.best_value

# Ejecutar
for dataset in split_data_dict.keys():
    X_train = split_data_dict[dataset]['train']['X']
    y_train = split_data_dict[dataset]['train']['Y'].ravel()
    print("\n==============================")
    print(f"Optimizando Random Forest para {dataset}")
    print("==============================\n")
    best_params, best_value = optimize_hyperparameters_rf(X_train, y_train)
    best_hyperparams["random_forest"][dataset] = {
        "best_params": best_params,
        "best_value": best_value
    }

# Detener monitor
monitor_flag = False
monitor_thread.join()

# Resultados
print("\n=== Mejores Hiperparámetros ===")
for dataset, result in best_hyperparams["random_forest"].items():
    print(f"\nDataset: {dataset}")
    print("Hiperparámetros:", result["best_params"])
    print(f"Mejor MSE: {result['best_value']:.5f}")

C:\Users\mjime\anaconda3\envs\tesiskarlo\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Optimizando Random Forest para orig



Best trial: 0. Best value: 20.4539:   0%|▏                                             | 1/200 [00:04<13:25,  4.05s/it]

Trial 0 | n_est=110 max_depth=45 | MSE: 20.45388 | Tiempo: 4.0s


Best trial: 0. Best value: 20.4539:   1%|▍                                             | 2/200 [00:08<13:46,  4.17s/it]

...

Best trial: 142. Best value: 21.2356: 100%|█████████████████████████████████████████▊| 199/200 [37:12<00:10, 10.59s/it]

Trial 198 | n_est=500 max_depth=5 | MSE: 99.59563 | Tiempo: 6.2s


Best trial: 142. Best value: 21.2356: 100%|██████████████████████████████████████████| 200/200 [37:25<00:00, 11.23s/it]

Trial 199 | n_est=490 max_depth=20 | MSE: 21.24047 | Tiempo: 13.1s



CPU Núcleos: [38.4, 39.9, 47.3, 50.0, 51.1, 48.8, 44.7, 40.4, 45.7, 44.8, 53.0, 48.7, 54.2, 52.7, 53.6, 52.3, 40.9, 43.8, 45.9, 49.6, 53.2, 53.7, 54.7, 53.1, 39.4, 46.0, 49.0, 50.6, 56.6, 55.5, 53.8, 56.1]
RAM Uso: 30.6%
GPU Uso: 0.0% | Mem: 4.6% | Temp: 51.0°C

CPU Núcleos: [0.1, 0.0, 0.0, 0.0, 0.0, 0.3, 0.2, 0.0, 0.0, 3.0, 1.8, 0.0, 0.2, 0.0, 0.0, 0.0, 0.1, 0.4, 0.8, 0.0, 0.5, 0.0, 0.1, 0.0, 0.7, 0.2, 0.1, 0.0, 0.4, 0.0, 0.0, 0.0]
RAM Uso: 30.6%
GPU Uso: 0.0% | Mem: 4.6% | Temp: 46.0°C

CPU Núcleos: [0.1, 0.0, 0.0, 0.0, 0.0, 0.1, 0.4, 0.0, 1.1, 2.3, 4.0, 0.0, 0.2, 0.0, 0.0, 0.0, 0.0, 0.2, 0.5, 0.2, 0.7, 0.2, 0.0, 0.0, 0.5, 0.2, 0.1, 0.0, 0.1, 0.1, 0.2, 0.1]
RAM Uso: 30.7%
GPU Uso: 0.0% | Mem: 4.6% | Temp: 45.0°C

CPU Núcleos: [0.1, 0.0, 0.0, 0.0, 0.1, 0.0, 0.2, 0.2, 2.1, 0.0, 3.6, 0.8, 0.5, 0.0, 0.2, 0.0, 0.2, 0.0, 0.3, 0.2, 0.8, 0.0, 0.0, 0.1, 0.2, 0.0, 0.0, 0.0, 0.5, 0.1, 0.0, 0.0]
RAM Uso: 30.9%
GPU Uso: 0.0% | Mem: 4.5% | Temp: 44.0°C

CPU Núcleos: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [14]:
import pickle
# suponiendo best_hyperparams ya lleno
filename = "best_hyperparams_all_models.pkl"
with open(filename, "wb") as f:
    pickle.dump(best_hyperparams, f)
